# Chat Supervised Learning: NoRobots SFT

This notebook demonstrates supervised fine-tuning (SFT) on conversational data using Tinker.

**Goal**: Fine-tune an LLM on the NoRobots dataset to improve conversational abilities.

**Expected Results**: Negative log-likelihood (NLL) decreases from ~2.5 to ~1.8 over training.

In [ ]:
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## Dataset: NoRobots

NoRobots is a high-quality conversational dataset with:
- 9,500 training examples
- 500 test examples
- Human-written responses (no GPT-generated content)

Categories include: brainstorming, creative writing, Q&A, summarization, etc.

In [ ]:
# Training configuration
config = {
    'model_name': 'meta-llama/Llama-3.2-1B',
    'dataset': 'no_robots',
    'learning_rate': 5e-4,
    'batch_size': 32,
    'lora_rank': 32,
    'eval_every': 10,
    'save_every': 10,
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## Run Training

In [ ]:
!python -m tinker_cookbook.recipes.chat_sl.train \
    model_name="meta-llama/Llama-3.2-1B" \
    dataset=no_robots \
    learning_rate=5e-4 \
    batch_size=32 \
    lora_rank=32 \
    eval_every=10 \
    save_every=10

## Understanding the Output

### Key Metrics
| Metric | Description |
|--------|-------------|
| `train_mean_nll` | Training negative log-likelihood (lower = better) |
| `test/nll` | Test set NLL (evaluated periodically) |
| `learning_rate` | Current learning rate (may have warmup) |
| `progress` | Training progress (0.0 to 1.0) |

### Sample Output
The training shows example conversations being learned:
```
User: Write a poem about...
Assistant: [Generated response]
```

## Expected Learning Curve

```
Step  10: train_nll=2.5, test_nll=2.4
Step  50: train_nll=2.2, test_nll=2.1
Step 100: train_nll=2.0, test_nll=1.9
Step 140: train_nll=1.8, test_nll=1.79
```

## Alternative: Tulu3 Dataset

For larger-scale training, use the Tulu3 dataset:

In [ ]:
# Tulu3 configuration (larger dataset, longer training)
tulu3_config = {
    'model_name': 'Qwen/Qwen3-8B-Base',
    'dataset': 'tulu3',
    'learning_rate': 5e-4,
    'batch_size': 128,
    'lora_rank': 64,
    'eval_every': 500,
    'save_every': 500,
}

print("Tulu3 config (for reference):")
for k, v in tulu3_config.items():
    print(f"  {k}: {v}")

## Analyze Results

In [ ]:
import json
import glob

# Find the latest metrics file
metrics_files = glob.glob('/tmp/tinker-examples/chat_sl/*/metrics.jsonl')
if metrics_files:
    latest = max(metrics_files)
    print(f"Reading: {latest}\n")
    
    nlls = []
    with open(latest) as f:
        for line in f:
            data = json.loads(line)
            if 'train_mean_nll' in data:
                nlls.append(data['train_mean_nll'])
    
    if nlls:
        print(f"Starting NLL: {nlls[0]:.4f}")
        print(f"Current NLL: {nlls[-1]:.4f}")
        print(f"Improvement: {nlls[0] - nlls[-1]:.4f}")